# Module 00: Method Selection Framework — Choosing the Right Tool

## Learning Objectives

By completing this notebook, you will:
1. Apply the decision framework from the taxonomy guide to real datasets by profiling problem characteristics
2. Implement a problem profiler that extracts the key inputs to the decision tree: p, n, p/n ratio, model type, and time budget
3. Run three selection families side-by-side on the same dataset and compare their results
4. Build a method selection function you can reuse across all later modules

## Prerequisites
- Notebooks 01 and 02 (landscape overview and benchmark datasets)
- Familiarity with sklearn selectors and cross-validation

## Estimated Time: 15 minutes

## Datasets

We use four datasets of varying scale — small, medium, high-dimensional, and time-constrained — to exercise each branch of the decision tree.

In [ ]:
import time
import warnings
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.datasets import (
    load_breast_cancer, load_wine, make_classification, fetch_openml
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import (
    SelectKBest, f_classif, mutual_info_classif,
    RFE, SelectFromModel, VarianceThreshold
)
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.dpi': 100,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

print('Imports complete.')

## Section 1: Problem Profiler

The decision framework requires knowing:
- **p** — total feature count
- **n** — sample count
- **p/n ratio** — signal of how hard the problem is statistically
- **model training time** — the key cost driver for wrapper and evolutionary methods
- **class balance** — affects MI and filter method reliability
- **feature types** — mixed data limits some methods

We build a `ProblemProfile` dataclass that captures all of these and a `profile_dataset` function that fills it automatically.

In [ ]:
@dataclass
class ProblemProfile:
    """
    Snapshot of the key properties of a feature selection problem.

    These properties drive method selection — they are the inputs
    to the decision framework.
    """
    n_samples: int
    n_features: int
    p_over_n: float
    model_train_time_sec: float   # single fit on the full dataset
    cv_train_time_sec: float      # 5-fold CV train time
    class_balance: float          # min(class_counts) / n_samples
    has_mixed_types: bool
    n_classes: int
    recommended_method: str = field(default='unknown')
    rationale: str = field(default='')

    def __str__(self):
        lines = [
            f'  n_samples          = {self.n_samples}',
            f'  n_features (p)     = {self.n_features}',
            f'  p / n ratio        = {self.p_over_n:.4f}',
            f'  T_model (1 fit)    = {self.model_train_time_sec*1000:.1f} ms',
            f'  T_5fold_CV         = {self.cv_train_time_sec:.3f} s',
            f'  class balance      = {self.class_balance:.3f}',
            f'  n_classes          = {self.n_classes}',
            f'  mixed types        = {self.has_mixed_types}',
            f'  recommended method = {self.recommended_method}',
            f'  rationale          = {self.rationale}',
        ]
        return '\n'.join(lines)


def profile_dataset(
    X: np.ndarray,
    y: np.ndarray,
    model=None,
    time_budget_sec: float = 300.0,
    has_mixed_types: bool = False,
) -> ProblemProfile:
    """
    Measure the key properties of a dataset and recommend a selection method.

    Parameters
    ----------
    X : np.ndarray (n, p)
    y : np.ndarray (n,)
    model : sklearn estimator — if None, uses LogisticRegression
    time_budget_sec : float — available wall-clock budget for selection
    has_mixed_types : bool — True if X contains categorical and numeric features

    Returns
    -------
    ProblemProfile
    """
    n, p = X.shape
    if model is None:
        model = LogisticRegression(max_iter=200, random_state=42)

    # -- Measure single-fit time
    t0 = time.perf_counter()
    model.fit(X, y)
    T_model = time.perf_counter() - t0

    # -- Measure 5-fold CV time
    t0 = time.perf_counter()
    cross_val_score(
        model, X, y, cv=5, scoring='accuracy', n_jobs=1
    )
    T_5cv = time.perf_counter() - t0

    # -- Class balance
    classes, counts = np.unique(y, return_counts=True)
    balance = counts.min() / n
    n_classes = len(classes)

    # -- p/n ratio
    p_over_n = p / n

    profile = ProblemProfile(
        n_samples=n,
        n_features=p,
        p_over_n=p_over_n,
        model_train_time_sec=T_model,
        cv_train_time_sec=T_5cv,
        class_balance=balance,
        has_mixed_types=has_mixed_types,
        n_classes=n_classes,
    )

    # -- Apply the decision framework
    profile.recommended_method, profile.rationale = (
        _apply_decision_framework(profile, time_budget_sec)
    )

    return profile


def _apply_decision_framework(
    prof: ProblemProfile,
    budget_sec: float
) -> tuple:
    """
    Implement the flowchart from the taxonomy guide (Guide 01, slide 11).

    Returns
    -------
    (method_name, rationale) : (str, str)
    """
    p = prof.n_features
    T_m = prof.model_train_time_sec

    # Estimate wrapper cost: forward selection k=20, p features, 5-fold CV
    # Approximate: k * p * f * T_m  (k=20, f=5)
    k_target = min(20, p // 5)
    wrapper_cost = k_target * p * 5 * T_m

    # Estimate embedded cost: 100 lambda values, 5-fold CV
    embedded_cost = 100 * 5 * T_m

    # Decision tree
    if p > 5000:
        if embedded_cost <= budget_sec / 2:
            return ('hybrid: filter + embedded',
                    f'p={p} > 5000 requires a filter pre-screen; '
                    f'embedded L1 path ({embedded_cost:.1f}s) fits the budget')
        return ('filter only',
                f'p={p} > 5000 and embedded cost ({embedded_cost:.1f}s) exceeds budget; '
                f'filter only')

    if p > 500:
        if wrapper_cost <= budget_sec:
            return ('hybrid: filter + wrapper',
                    f'p={p} is large; filter pre-screens to p\'<200, '
                    f'then wrapper (estimated {wrapper_cost:.1f}s)')
        return ('embedded',
                f'p={p}: wrapper cost ({wrapper_cost:.1f}s) exceeds budget ({budget_sec}s); '
                f'use embedded')

    # Moderate p (<= 500)
    if wrapper_cost <= budget_sec:
        return ('wrapper',
                f'p={p} <= 500 and wrapper cost ({wrapper_cost:.1f}s) <= budget; '
                f'use wrapper for best solution quality')

    if embedded_cost <= budget_sec:
        return ('embedded',
                f'p={p}: wrapper too slow ({wrapper_cost:.1f}s); '
                f'embedded is feasible ({embedded_cost:.1f}s)')

    return ('filter',
            f'Both wrapper ({wrapper_cost:.1f}s) and embedded ({embedded_cost:.1f}s) '
            f'exceed budget ({budget_sec}s); use filter')


print('ProblemProfile and profiler defined.')

## Section 2: Profile Four Datasets

We profile four datasets that represent different regimes in the decision framework:

| Dataset | n | p | p/n | Expected recommendation |
|---------|---|---|-----|------------------------|
| Breast cancer | 569 | 30 | 0.05 | Wrapper (fast model, small p) |
| Wine | 178 | 13 | 0.07 | Wrapper or filter (very small) |
| Synthetic medium | 500 | 200 | 0.40 | Depends on budget |
| Synthetic high-dim | 300 | 1000 | 3.33 | Hybrid or embedded |

In [ ]:
def load_datasets():
    """Load four datasets for profiling."""
    scaler = StandardScaler()
    datasets = {}

    # Breast cancer (n=569, p=30)
    bc = load_breast_cancer()
    datasets['breast_cancer'] = (
        scaler.fit_transform(bc.data), bc.target
    )

    # Wine (n=178, p=13)
    wine = load_wine()
    # Binarise wine for binary classification
    y_wine = (wine.target == 0).astype(int)
    datasets['wine_binary'] = (
        scaler.fit_transform(wine.data), y_wine
    )

    # Synthetic medium (n=500, p=200)
    X_med, y_med = make_classification(
        n_samples=500, n_features=200, n_informative=20,
        n_redundant=30, random_state=42
    )
    datasets['synthetic_medium'] = (
        scaler.fit_transform(X_med), y_med
    )

    # Synthetic high-dim (n=300, p=1000)
    X_hd, y_hd = make_classification(
        n_samples=300, n_features=1000, n_informative=15,
        n_redundant=50, random_state=42
    )
    datasets['synthetic_highdim'] = (
        scaler.fit_transform(X_hd), y_hd
    )

    return datasets


datasets = load_datasets()
print('Loaded datasets:')
for name, (X, y) in datasets.items():
    print(f'  {name}: n={X.shape[0]}, p={X.shape[1]}, '
          f'p/n={X.shape[1]/X.shape[0]:.3f}')

In [ ]:
# Profile each dataset with a 60-second time budget
profiles = {}

for name, (X, y) in datasets.items():
    print(f'Profiling {name}...')
    model = LogisticRegression(max_iter=300, random_state=42)
    profiles[name] = profile_dataset(
        X, y, model=model, time_budget_sec=60.0
    )
    print(f'\n{name}:')
    print(profiles[name])
    print()

In [ ]:
# Summarise all profiles in a comparison table
summary_rows = []
for name, prof in profiles.items():
    summary_rows.append({
        'dataset': name,
        'n': prof.n_samples,
        'p': prof.n_features,
        'p/n': f'{prof.p_over_n:.3f}',
        'T_model (ms)': f'{prof.model_train_time_sec * 1000:.1f}',
        'recommended': prof.recommended_method,
    })

summary_df = pd.DataFrame(summary_rows)
print('Profile Summary:')
print(summary_df.to_string(index=False))

## Section 3: Three Families Side-by-Side

We now apply one representative method from each family to the **synthetic_medium** dataset (n=500, p=200) and compare:
- Downstream accuracy (5-fold CV logistic regression)
- Number of features selected
- Wall-clock time for the selection step

This is the core comparison we will revisit at the end of every module.

In [ ]:
def run_three_families(
    X: np.ndarray,
    y: np.ndarray,
    k_target: int = 20,
    cv: int = 5,
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Run filter, wrapper, and embedded selection on the same dataset.

    Evaluates downstream accuracy for each selection strategy.

    Parameters
    ----------
    X : np.ndarray (n, p) — standardised
    y : np.ndarray (n,)
    k_target : int — approximate target feature count for filter and wrapper
    cv : int — CV folds for downstream evaluation
    random_state : int

    Returns
    -------
    pd.DataFrame with columns:
        family, method, n_selected, selection_time_s, accuracy_mean, accuracy_std
    """
    downstream = LogisticRegression(max_iter=300, random_state=random_state)
    cv_splitter = StratifiedKFold(n_splits=cv, shuffle=True,
                                   random_state=random_state)
    results = []

    def evaluate(X_sel, label, family):
        scores = cross_val_score(
            downstream, X_sel, y, cv=cv_splitter, scoring='accuracy'
        )
        results.append({
            'family': family,
            'method': label,
            'n_selected': X_sel.shape[1],
            'accuracy_mean': scores.mean(),
            'accuracy_std': scores.std(),
        })

    # -- Baseline: all features
    t0 = time.perf_counter()
    X_all = X
    t_sel = time.perf_counter() - t0
    scores = cross_val_score(downstream, X_all, y, cv=cv_splitter, scoring='accuracy')
    results.append({
        'family': 'baseline',
        'method': f'All {X.shape[1]} features',
        'n_selected': X.shape[1],
        'accuracy_mean': scores.mean(),
        'accuracy_std': scores.std(),
    })

    # -- Filter: MI-based SelectKBest
    t0 = time.perf_counter()
    mi_selector = SelectKBest(mutual_info_classif, k=k_target)
    X_filter = mi_selector.fit_transform(X, y)
    t_filter = time.perf_counter() - t0
    scores = cross_val_score(downstream, X_filter, y, cv=cv_splitter, scoring='accuracy')
    results.append({
        'family': 'filter',
        'method': f'MI filter (k={k_target})',
        'n_selected': X_filter.shape[1],
        'selection_time_s': t_filter,
        'accuracy_mean': scores.mean(),
        'accuracy_std': scores.std(),
    })

    # -- Wrapper: RFE
    t0 = time.perf_counter()
    rfe_estimator = LogisticRegression(max_iter=200, random_state=random_state)
    rfe = RFE(rfe_estimator, n_features_to_select=k_target, step=5)
    X_wrapper = rfe.fit_transform(X, y)
    t_wrapper = time.perf_counter() - t0
    scores = cross_val_score(downstream, X_wrapper, y, cv=cv_splitter, scoring='accuracy')
    results.append({
        'family': 'wrapper',
        'method': f'RFE (k={k_target}, step=5)',
        'n_selected': X_wrapper.shape[1],
        'selection_time_s': t_wrapper,
        'accuracy_mean': scores.mean(),
        'accuracy_std': scores.std(),
    })

    # -- Embedded: L1 Logistic Regression
    t0 = time.perf_counter()
    l1_model = LogisticRegression(
        penalty='l1', solver='liblinear', C=0.5,
        max_iter=300, random_state=random_state
    )
    emb_selector = SelectFromModel(l1_model, threshold=1e-5)
    X_embedded = emb_selector.fit_transform(X, y)
    t_embedded = time.perf_counter() - t0
    scores = cross_val_score(downstream, X_embedded, y, cv=cv_splitter, scoring='accuracy')
    results.append({
        'family': 'embedded',
        'method': f'L1 Logistic (C=0.5)',
        'n_selected': X_embedded.shape[1],
        'selection_time_s': t_embedded,
        'accuracy_mean': scores.mean(),
        'accuracy_std': scores.std(),
    })

    df = pd.DataFrame(results)
    df['selection_time_s'] = df.get('selection_time_s', 0).fillna(0)
    return df


X_med, y_med = datasets['synthetic_medium']
print(f'Running three-family comparison on synthetic_medium '
      f'(n={X_med.shape[0]}, p={X_med.shape[1]})...')
comparison_df = run_three_families(X_med, y_med, k_target=20)

print('\nResults:')
print(comparison_df[[
    'family', 'method', 'n_selected', 'accuracy_mean', 'accuracy_std'
]].to_string(index=False))

In [ ]:
# Visualise the three-family comparison
family_colours = {
    'baseline': '#9E9E9E',
    'filter':   '#4CAF50',
    'wrapper':  '#FF9800',
    'embedded': '#2196F3',
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left panel: downstream accuracy
ax_acc = axes[0]
colours = [family_colours[fam] for fam in comparison_df['family']]
bars = ax_acc.bar(
    range(len(comparison_df)),
    comparison_df['accuracy_mean'],
    color=colours,
    yerr=comparison_df['accuracy_std'],
    capsize=4,
    alpha=0.85,
    edgecolor='white',
)

for bar, val in zip(bars, comparison_df['accuracy_mean']):
    ax_acc.text(
        bar.get_x() + bar.get_width() / 2,
        val + 0.005,
        f'{val:.3f}',
        ha='center', va='bottom', fontsize=9, fontweight='bold'
    )

labels = [f"{row['family']}\n({row['n_selected']} feat)"
          for _, row in comparison_df.iterrows()]
ax_acc.set_xticks(range(len(comparison_df)))
ax_acc.set_xticklabels(labels, fontsize=9)
ax_acc.set_ylabel('5-fold CV Accuracy', fontsize=11)
ax_acc.set_title('Downstream Accuracy\nby Selection Family', fontsize=12)
y_range = comparison_df['accuracy_mean']
ax_acc.set_ylim(y_range.min() - 0.03, y_range.max() + 0.05)
ax_acc.grid(True, alpha=0.3, axis='y')

# Right panel: n_selected vs accuracy scatter
ax_scatter = axes[1]
for _, row in comparison_df.iterrows():
    c = family_colours[row['family']]
    ax_scatter.scatter(
        row['n_selected'], row['accuracy_mean'],
        color=c, s=150, zorder=5, edgecolors='black', linewidth=0.5
    )
    ax_scatter.annotate(
        row['family'],
        (row['n_selected'], row['accuracy_mean']),
        textcoords='offset points', xytext=(6, 4), fontsize=9
    )

ax_scatter.set_xlabel('Number of Features Selected', fontsize=11)
ax_scatter.set_ylabel('5-fold CV Accuracy', fontsize=11)
ax_scatter.set_title('Accuracy vs. Features Selected\n(Smaller & Higher = Better)', fontsize=12)
ax_scatter.grid(True, alpha=0.3)

# Legend
patches = [mpatches.Patch(color=c, label=fam)
           for fam, c in family_colours.items()]
ax_scatter.legend(handles=patches, fontsize=9, loc='lower right')

fig.suptitle(
    'Three Selection Families — Accuracy vs. Parsimony\n'
    f'synthetic_medium: n={X_med.shape[0]}, p={X_med.shape[1]}, k_target=20',
    fontsize=13, y=1.02
)

plt.tight_layout()
plt.savefig('three_families_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nAnnotation:')
for _, row in comparison_df.iterrows():
    print(f"  {row['family']:10s}: {row['n_selected']:3d} features, "
          f"acc={row['accuracy_mean']:.4f}")

## Section 4: Reusable Method Selector

We combine the profiler and three-family runner into a single `select_features` function. This function:
1. Profiles the dataset automatically
2. Chooses the appropriate method family
3. Runs the selection and returns the selected feature indices

This function becomes your starting point for all future modules.

In [ ]:
def select_features(
    X: np.ndarray,
    y: np.ndarray,
    k_target: int = 20,
    time_budget_sec: float = 60.0,
    method: Optional[str] = None,
    random_state: int = 42,
    verbose: bool = True,
) -> tuple:
    """
    Select features from X using the most appropriate method given the problem.

    If method is None, auto-selects based on the decision framework.
    If method is provided, uses that family directly.

    Parameters
    ----------
    X : np.ndarray (n, p) — standardised
    y : np.ndarray (n,)
    k_target : int — approximate number of features to select
    time_budget_sec : float — allowed wall-clock time
    method : str or None — one of 'filter', 'wrapper', 'embedded', 'auto'
    random_state : int
    verbose : bool — print selection summary

    Returns
    -------
    (selected_indices, profile) : (np.ndarray, ProblemProfile)
    """
    # Profile the dataset
    lr = LogisticRegression(max_iter=200, random_state=random_state)
    prof = profile_dataset(X, y, model=lr, time_budget_sec=time_budget_sec)

    chosen_method = method if method else prof.recommended_method

    if verbose:
        print(f'Dataset profile: n={prof.n_samples}, p={prof.n_features}, '
              f'p/n={prof.p_over_n:.3f}')
        print(f'Method chosen: {chosen_method}')
        print(f'Rationale: {prof.rationale}')

    t0 = time.perf_counter()

    if 'filter' in chosen_method:
        k = min(k_target, X.shape[1])
        selector = SelectKBest(mutual_info_classif, k=k)
        selector.fit(X, y)
        indices = np.where(selector.get_support())[0]

    elif 'wrapper' in chosen_method:
        estimator = LogisticRegression(max_iter=200, random_state=random_state)
        k = min(k_target, X.shape[1])
        rfe = RFE(estimator, n_features_to_select=k, step=max(1, X.shape[1] // 20))
        rfe.fit(X, y)
        indices = np.where(rfe.support_)[0]

    else:  # embedded (default)
        l1_lr = LogisticRegression(
            penalty='l1', solver='liblinear', C=0.5,
            max_iter=300, random_state=random_state
        )
        emb = SelectFromModel(l1_lr, threshold=1e-5)
        emb.fit(X, y)
        indices = np.where(emb.get_support())[0]

    t_sel = time.perf_counter() - t0

    if verbose:
        print(f'Selected {len(indices)} features in {t_sel:.3f}s')

    return indices, prof


# Demo: run select_features on each dataset
print('Demonstrating select_features on all four datasets:\n')
print('=' * 65)

for name, (X, y) in datasets.items():
    print(f'\n--- {name} ---')
    idx, prof = select_features(
        X, y, k_target=15, time_budget_sec=60.0, verbose=True
    )
    print(f'Selected indices (first 10): {idx[:10]}')

## Section 5: Stability of the Framework Across Time Budgets

The decision framework is time-budget-sensitive. Here we sweep the budget and observe how the recommended method changes as we allow more (or less) time.

This shows that the framework is not rigid — as time budget grows, it graduates from filter to embedded to wrapper.

In [ ]:
X_test, y_test = datasets['synthetic_medium']
lr_probe = LogisticRegression(max_iter=200, random_state=42)

# Sweep time budgets from 0.1 seconds to 1 hour
budgets = [0.1, 1.0, 5.0, 30.0, 60.0, 300.0, 3600.0]
method_by_budget = []

for budget in budgets:
    prof = profile_dataset(X_test, y_test, model=lr_probe, time_budget_sec=budget)
    method_by_budget.append({
        'budget_sec': budget,
        'budget_label': f'{budget:.1f}s' if budget < 60 else f'{budget/60:.0f}min',
        'recommended': prof.recommended_method,
    })

budget_df = pd.DataFrame(method_by_budget)
print('Method recommendation by time budget:')
print(f'{"Budget":>10}  Recommended Method')
print('-' * 50)
for _, row in budget_df.iterrows():
    print(f'{row["budget_label"]:>10}  {row["recommended"]}')

## Summary

### Key Takeaways

1. **Profile before you select.** The three key inputs — p, n, and T_model — determine which method family is feasible within your time budget. Always profile first.

2. **The decision framework is a decision tree, not a lookup table.** The same dataset can warrant different methods depending on the time budget. With 5 seconds, use a filter. With 5 minutes, use embedded. With 1 hour, use a wrapper.

3. **All three families can produce competitive accuracy.** On the synthetic medium dataset, filter, wrapper, and embedded all reach within a few percent of each other. The real differentiator is compute cost, not accuracy (for well-chosen hyperparameters).

4. **The `select_features` function is your scaffold.** Reuse it in later modules — swap the internal selector to the method of the day (mRMR, Boruta, GA) while keeping the profiling and evaluation wrapper unchanged.

### What's Next

- **Module 01:** Statistical filter methods in depth — mutual information, ReliefF, and mRMR from scratch.
- **Module 03:** Wrapper algorithms — sequential search and Boruta with scalability tricks.
- **Module 04:** Embedded methods — Lasso paths, stability selection, and tree-based importances.

### Going Further

- Replace `LogisticRegression` with `RandomForestClassifier` in the profiler and observe how `T_model` changes the recommendation.
- Add a fourth panel to the three-family comparison for an evolutionary method (GA) — Module 05 implements this.
- Try `select_features` on your own dataset with `method='auto'`.